In [1]:
import numpy as np
import pandas as pd
from scipy.signal import welch

In [2]:
windows = np.load("../data/windows.npy")
labels = np.load("../data/labels.npy")

print("Windows Shape:", windows.shape)
print("Labels Shape:", labels.shape)

Windows Shape: (900, 23, 1024)
Labels Shape: (900,)


In [3]:
print("Total Windows:", len(windows))
print("Total Labels:", len(labels))

print("\nClass Distribution:")
print("Normal Windows:", np.sum(labels == 0))
print("Seizure Windows:", np.sum(labels == 1))

Total Windows: 900
Total Labels: 900

Class Distribution:
Normal Windows: 890
Seizure Windows: 10


In [4]:
def extract_statistical_features(window):
    
    # Calculate features for each EEG channel
    channel_means = np.mean(window, axis=1)
    channel_stds = np.std(window, axis=1)
    channel_variances = np.var(window, axis=1)
    
    # Average across all EEG channels
    mean_feature = np.mean(channel_means)
    std_feature = np.mean(channel_stds)
    variance_feature = np.mean(channel_variances)
    
    return mean_feature, std_feature, variance_feature

In [5]:
mean_value, std_value, variance_value = extract_statistical_features(windows[0])

print("Mean:", mean_value)
print("Standard Deviation:", std_value)
print("Variance:", variance_value)

Mean: 6.224838260383522e-06
Standard Deviation: 3.986366016444262e-05
Variance: 2.0190720572477735e-09


In [6]:
frequency_bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 40)
}

print("Frequency Bands:")
for band, (low, high) in frequency_bands.items():
    print(f"{band}: {low}-{high} Hz")

Frequency Bands:
Delta: 0.5-4 Hz
Theta: 4-8 Hz
Alpha: 8-13 Hz
Beta: 13-30 Hz
Gamma: 30-40 Hz


In [7]:
sfreq = 256

print("Sampling Frequency:", sfreq, "Hz")

Sampling Frequency: 256 Hz


In [8]:
def extract_frequency_features(window, sfreq=256):
    
    # Store frequency features for all EEG channels
    channel_features = []
    
    # Process each EEG channel
    for channel in window:
        
        # Calculate Power Spectral Density using Welch method
        frequencies, psd = welch(
            channel,
            fs=sfreq,
            nperseg=512
        )
        
        # Store features for this channel
        band_features = {}
        
        # Calculate power in each frequency band
        for band_name, (low_freq, high_freq) in frequency_bands.items():
            
            # Select frequencies within the band
            frequency_mask = (
                (frequencies >= low_freq) &
                (frequencies < high_freq)
            )
            
            # Calculate band power
            band_power = np.trapezoid(
                psd[frequency_mask],
                frequencies[frequency_mask]
            )
            
            band_features[band_name] = band_power
        
        channel_features.append(band_features)
    
    # Convert to DataFrame
    channel_features_df = pd.DataFrame(channel_features)
    
    # Average frequency features across all 23 EEG channels
    window_frequency_features = channel_features_df.mean()
    
    return window_frequency_features

In [9]:
frequency_features = extract_frequency_features(windows[0], sfreq)

print(frequency_features)

Delta    5.086776e-10
Theta    1.252318e-10
Alpha    4.501589e-11
Beta     9.936528e-11
Gamma    5.169968e-11
dtype: float64


In [10]:
def extract_all_features(window, sfreq=256):
    
    # -----------------------------
    # Statistical Features
    # -----------------------------
    
    mean_feature, std_feature, variance_feature = extract_statistical_features(window)
    
    
    # -----------------------------
    # Frequency-Domain Features
    # -----------------------------
    
    frequency_features = extract_frequency_features(window, sfreq)
    
    
    # -----------------------------
    # Combine All Features
    # -----------------------------
    
    return [
        mean_feature,
        std_feature,
        variance_feature,
        frequency_features["Delta"],
        frequency_features["Theta"],
        frequency_features["Alpha"],
        frequency_features["Beta"],
        frequency_features["Gamma"]
    ]

In [11]:
test_features = extract_all_features(windows[0], sfreq)

print("Extracted Features:")
print(test_features)

print("\nNumber of Features:", len(test_features))

Extracted Features:
[np.float64(6.224838260383522e-06), np.float64(3.986366016444262e-05), np.float64(2.0190720572477735e-09), np.float64(5.086775512929678e-10), np.float64(1.252317849986049e-10), np.float64(4.5015894546181256e-11), np.float64(9.936527828679301e-11), np.float64(5.169968196195911e-11)]

Number of Features: 8


In [12]:
all_features = []

for i, window in enumerate(windows):
    
    features_for_window = extract_all_features(window, sfreq)
    
    all_features.append(features_for_window)
    
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(windows)} windows")

Processed 100/900 windows
Processed 200/900 windows
Processed 300/900 windows
Processed 400/900 windows
Processed 500/900 windows
Processed 600/900 windows
Processed 700/900 windows
Processed 800/900 windows
Processed 900/900 windows


In [13]:
features = np.array(all_features)

print("Feature Matrix Shape:", features.shape)

Feature Matrix Shape: (900, 8)


In [14]:
feature_names = [
    "Mean",
    "Std",
    "Variance",
    "Delta",
    "Theta",
    "Alpha",
    "Beta",
    "Gamma"
]

features_df = pd.DataFrame(
    features,
    columns=feature_names
)

features_df.head()

,Mean,Std,Variance,Delta,Theta,Alpha,Beta,Gamma
0,6.224838e-06,0.000040,2.019072e-09,5.086776e-10,1.252318e-10,4.501589e-11,9.936528e-11,5.169968e-11
1,-6.238922e-07,0.000035,1.604471e-09,1.510739e-09,1.018136e-10,3.810161e-11,1.113378e-10,7.269863e-11
2,9.433753e-07,0.000047,3.111549e-09,2.225362e-09,2.902292e-10,5.649391e-11,1.265199e-10,9.276142e-11
3,2.200194e-07,0.000031,1.184050e-09,6.634073e-10,1.910103e-10,4.744904e-11,2.174806e-10,1.205579e-10
4,-1.851659e-07,0.000025,7.406582e-10,3.636297e-10,9.952621e-11,3.020521e-11,1.349834e-10,7.228049e-11


In [15]:
features_df["Label"] = labels

print("Final Dataset Shape:", features_df.shape)

Final Dataset Shape: (900, 9)


In [16]:
features_df.head()

,Mean,Std,Variance,Delta,Theta,Alpha,Beta,Gamma,Label
0,6.224838e-06,0.000040,2.019072e-09,5.086776e-10,1.252318e-10,4.501589e-11,9.936528e-11,5.169968e-11,0
1,-6.238922e-07,0.000035,1.604471e-09,1.510739e-09,1.018136e-10,3.810161e-11,1.113378e-10,7.269863e-11,0
2,9.433753e-07,0.000047,3.111549e-09,2.225362e-09,2.902292e-10,5.649391e-11,1.265199e-10,9.276142e-11,0
3,2.200194e-07,0.000031,1.184050e-09,6.634073e-10,1.910103e-10,4.744904e-11,2.174806e-10,1.205579e-10,0
4,-1.851659e-07,0.000025,7.406582e-10,3.636297e-10,9.952621e-11,3.020521e-11,1.349834e-10,7.228049e-11,0


In [17]:
print("Final Class Distribution:")

print(
    features_df["Label"].value_counts().sort_index()
)

Final Class Distribution:
Label
0    890
1     10
Name: count, dtype: int64


In [18]:
print("Missing Values:")
print(features_df.isnull().sum())

Missing Values:
Mean        0
Std         0
Variance    0
Delta       0
Theta       0
Alpha       0
Beta        0
Gamma       0
Label       0
dtype: int64


In [19]:
features_df.to_csv(
    "../data/features.csv",
    index=False
)

print("Feature dataset saved successfully!")

Feature dataset saved successfully!


In [20]:
np.save("../data/features.npy", features)
np.save("../data/labels.npy", labels)

print("Features and labels saved successfully!")

Features and labels saved successfully!


In [21]:
print("========== FINAL FEATURE DATASET CHECK ==========")

print("Features Shape:", features.shape)
print("Labels Shape:", labels.shape)

print("\nClass Distribution:")
print("Normal:", np.sum(labels == 0))
print("Seizure:", np.sum(labels == 1))

print("\nFeature Names:")
print(feature_names)

print("\nFinal DataFrame Shape:", features_df.shape)

print("\nFirst 5 Rows:")
display(features_df.head())

========== FINAL FEATURE DATASET CHECK ==========
Features Shape: (900, 8)
Labels Shape: (900,)

Class Distribution:
Normal: 890
Seizure: 10

Feature Names:
['Mean', 'Std', 'Variance', 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']

Final DataFrame Shape: (900, 9)

First 5 Rows:


,Mean,Std,Variance,Delta,Theta,Alpha,Beta,Gamma,Label
0,6.224838e-06,0.000040,2.019072e-09,5.086776e-10,1.252318e-10,4.501589e-11,9.936528e-11,5.169968e-11,0
1,-6.238922e-07,0.000035,1.604471e-09,1.510739e-09,1.018136e-10,3.810161e-11,1.113378e-10,7.269863e-11,0
2,9.433753e-07,0.000047,3.111549e-09,2.225362e-09,2.902292e-10,5.649391e-11,1.265199e-10,9.276142e-11,0
3,2.200194e-07,0.000031,1.184050e-09,6.634073e-10,1.910103e-10,4.744904e-11,2.174806e-10,1.205579e-10,0
4,-1.851659e-07,0.000025,7.406582e-10,3.636297e-10,9.952621e-11,3.020521e-11,1.349834e-10,7.228049e-11,0
